In [0]:
from pyspark.sql.functions import (
    col, hour, dayofweek, count, countDistinct, 
    when, lit, round, avg
)

DELTA_VOLUME_PATH = "/Volumes/bc_transit_ws/gtfs/delta_volume/"
GOLD_BASE_PATH = f"{DELTA_VOLUME_PATH}gold/"
GOLD_REALTIME_PATH = f"{DELTA_VOLUME_PATH}realtime_gold/live_positions"

CATALOG_NAME = "bc_transit_ws"
SCHEMA_NAME = "gold"

print("Starting Route Reliability Index calculation...")

In [0]:
# Load static Gold fact table (scheduled trips)
df_scheduled = spark.read.format("delta").load(f"{GOLD_BASE_PATH}fact_trip_schedule/")

# Load realtime Gold table (observed vehicles)
df_observed = spark.read.format("delta").load(GOLD_REALTIME_PATH)

print(f"Scheduled rows: {df_scheduled.count():,}")
print(f"Observed rows:  {df_observed.count():,}")

In [0]:
from pyspark.sql.functions import min as spark_min, max as spark_max

# Count scheduled weekdays and weekends in the observation window
# by looking at what days of the week are represented in realtime data
df_day_counts = df_observed \
    .withColumn("day_of_week", dayofweek(col("ingestion_timestamp"))) \
    .withColumn(
        "day_type",
        when(col("day_of_week").isin([1, 7]), lit("weekend"))
        .otherwise(lit("weekday"))
    ) \
    .select(
        "day_type",
        col("ingestion_timestamp").cast("date").alias("date")
    ) \
    .distinct() \
    .groupBy("day_type") \
    .agg(countDistinct("date").alias("days_observed"))

days_observed = {
    row["day_type"]: row["days_observed"]
    for row in df_day_counts.collect()
}

weekdays_observed = days_observed.get("weekday", 1)
weekends_observed = days_observed.get("weekend", 1)

print(f"Weekdays observed: {weekdays_observed}")
print(f"Weekend days observed: {weekends_observed}")

# Build scheduled trips per typical day type and time bucket
# monday=1 means this service runs on Mondays — representative weekday
# saturday=1 or sunday=1 means weekend service
df_scheduled_hourly = df_scheduled \
    .filter(col("arrival_time").isNotNull()) \
    .withColumn(
        "hour_of_day",
        col("arrival_time").substr(1, 2).cast("integer")
    ) \
    .withColumn(
        "day_type",
        when(
            (col("saturday") == 1) | (col("sunday") == 1),
            lit("weekend")
        ).otherwise(lit("weekday"))
    ) \
    .withColumn(
        "time_bucket",
        when(col("day_type") == "weekend", lit("weekend"))
        .when(
            col("hour_of_day").isin([7, 8, 16, 17, 18]),
            lit("weekday_peak")
        )
        .otherwise(lit("weekday_offpeak"))
    ) \
    .groupBy("route_short_name_display", "time_bucket", "day_type") \
    .agg(countDistinct("trip_id").alias("total_scheduled_trips")) \
    .withColumn(
        "scheduled_trips_per_day",
        when(
            col("day_type") == "weekend",
            round(col("total_scheduled_trips") / weekends_observed, 1)
        ).otherwise(
            round(col("total_scheduled_trips") / weekdays_observed, 1)
        )
    )

print("Scheduled trips per typical day:")
df_scheduled_hourly.display(10)

In [0]:

df_observed_hourly = df_observed \
    .withColumn("hour_of_day", hour(col("ingestion_timestamp"))) \
    .withColumn("day_of_week", dayofweek(col("ingestion_timestamp"))) \
    .withColumn(
        "day_type",
        when(col("day_of_week").isin([1, 7]), lit("weekend"))
        .otherwise(lit("weekday"))
    ) \
    .withColumn(
        "time_bucket",
        when(col("day_type") == "weekend", lit("weekend"))
        .when(
            col("hour_of_day").isin([7, 8, 16, 17, 18]),
            lit("weekday_peak")
        )
        .otherwise(lit("weekday_offpeak"))
    ) \
    .groupBy("route_name", "time_bucket", "day_type") \
    .agg(countDistinct("trip_id").alias("total_observed_trips")) \
    .withColumn(
        "observed_trips_per_day",
        when(
            col("day_type") == "weekend",
            round(col("total_observed_trips") / weekends_observed, 1)
        ).otherwise(
            round(col("total_observed_trips") / weekdays_observed, 1)
        )
    )

print("Observed trips per typical day:")
df_observed_hourly.display(10)


In [0]:
df_coverage = df_scheduled_hourly.alias("s") \
    .join(
        df_observed_hourly.alias("o"),
        on=(
            (col("s.route_short_name_display") == col("o.route_name")) &
            (col("s.time_bucket") == col("o.time_bucket"))
        ),
        how="inner"
    ) \
    .select(
        col("s.route_short_name_display").alias("route_name"),
        col("s.time_bucket"),
        col("s.scheduled_trips_per_day"),
        col("o.observed_trips_per_day"),
        round(
            (col("o.observed_trips_per_day") / col("s.scheduled_trips_per_day")) * 100,
            2
        ).alias("coverage_pct")
    ) \
    .filter(col("coverage_pct") <= 100)

print("Coverage ratios per route per time bucket:")
df_coverage.orderBy(col("coverage_pct").desc()).display(20)

In [0]:
# Join scheduled vs observed on route name and time bucket
df_coverage = df_scheduled_hourly.alias("s") \
    .join(
        df_observed_hourly.alias("o"),
        on=(
            (col("s.route_short_name_display") == col("o.route_name")) &
            (col("s.time_bucket") == col("o.time_bucket"))
        ),
        how="inner"
    ) \
    .select(
        col("s.route_short_name_display").alias("route_name"),
        col("s.time_bucket"),
        col("s.scheduled_trips_per_day"),
        col("o.observed_trips_per_day"),
        round(
            (col("o.observed_trips_per_day") / col("s.scheduled_trips_per_day")) * 100,
            2
        ).alias("coverage_pct")
    ) \
    .filter(col("coverage_pct") <= 100)

print("Coverage ratios per route per time bucket:")
df_coverage.orderBy(col("coverage_pct").desc()).display(20)

print(f"Weekdays observed: {weekdays_observed}")
print(f"Weekend days observed: {weekends_observed}")


In [0]:
from pyspark.sql.functions import sum as spark_sum, coalesce

df_pivot = df_coverage \
    .groupBy("route_name") \
    .pivot("time_bucket", ["weekday_peak", "weekday_offpeak", "weekend"]) \
    .agg(round(avg("coverage_pct"), 2))

# Use coalesce to default missing buckets to 0
df_reliability = df_pivot \
    .withColumn(
        "reliability_score",
        round(
            (
                coalesce(col("weekday_peak"), lit(0.0))    * 0.40 +
                coalesce(col("weekday_offpeak"), lit(0.0)) * 0.35 +
                coalesce(col("weekend"), lit(0.0))         * 0.25
            ),
            2
        )
    ) \
    .select(
        col("route_name"),
        coalesce(col("weekday_peak"), lit(0.0)).alias("peak_coverage_pct"),
        coalesce(col("weekday_offpeak"), lit(0.0)).alias("offpeak_coverage_pct"),
        coalesce(col("weekend"), lit(0.0)).alias("weekend_coverage_pct"),
        col("reliability_score")
    ) \
    .filter(col("reliability_score") > 0) \
    .orderBy(col("reliability_score").desc())

print("Route Reliability Index:")
df_reliability.display(30)

In [0]:
df_check = spark.read.format("delta").load(GOLD_REALTIME_PATH)

from pyspark.sql.functions import date_format
df_check.groupBy(date_format("ingestion_timestamp", "yyyy-MM-dd").alias("date")) \
    .count() \
    .orderBy("date") \
    .display()